In [1]:
# --- الخلية 1: التثبيت الذكي (Smart Offline Install) ---
import sys
import os
import glob
import subprocess

print("⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...")

# 1. تحديد المكتبات الخطرة التي لا يجب إعادة تثبيتها لأنها موجودة ومجهزة للـ GPU
FORBIDDEN_PACKAGES = [
    "torch-", "torchvision", "torchaudio",  # نترك نسخ Kaggle الأصلية
    "numpy", "pandas", "opencv", "matplotlib", "scipy", "pillow" # موجودة مسبقاً
]

# 2. العثور على الملفات
all_whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
print(f"📦 تم العثور على {len(all_whls)} ملف في الداتا سيت.")

success_count = 0
for path in all_whls:
    filename = os.path.basename(path).lower()
    
    # التحقق: هل هذا الملف "ممنوع"؟
    is_forbidden = False
    for forbidden in FORBIDDEN_PACKAGES:
        if forbidden in filename:
            is_forbidden = True
            break
    
    if is_forbidden:
        # print(f"🛡️ تم الحفاظ على نسخة النظام من: {filename}")
        continue

    # التثبيت الآمن للمكتبات المفقودة فقط (مثل YOLO, SMP)
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            path, 
            "--no-deps",
            "--ignore-installed"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        success_count += 1
        # print(f"✅ تم تثبيت: {filename}")
    except Exception as e:
        print(f"⚠️ فشل تثبيت: {filename}")

print(f"✅ تمت العملية! تم تثبيت {success_count} مكتبة إضافية.")
print("⚡ إعدادات الـ Torch الحالية:")
import torch
print(f"   - Torch Version: {torch.__version__}")
print(f"   - CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ تنبيه خطير: الـ GPU غير مفعل! تأكد من إعدادات Kaggle Sidebar.")

# إضافة مسارات YOLO إذا لزم الأمر
src_dirs = glob.glob("/kaggle/input/**/ultralytics", recursive=True)
for d in src_dirs:
    parent = os.path.dirname(d)
    if parent not in sys.path:
        sys.path.append(parent)

⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...
📦 تم العثور على 95 ملف في الداتا سيت.
⚠️ فشل تثبيت: pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.5.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.0.0-1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: kiwisolver-1.4.5-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl
⚠️ فشل تثبيت: markupsafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: fonttools-4.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: charset_normalizer-3.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: contourpy-1.2.1-cp310-cp310-manylinux_2_17_x

In [2]:
# --- Cell 2: Offline install/import fix (NO internet) ---
import sys, os, glob, subprocess, site
from importlib import reload

print("🔧 Cell 2: Offline install/import fix (no internet).")

# تعطيل أي شيء ممكن يحاول يتصل بالنت
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def ensure_pkg(import_name: str, wheel_hint: str):
    """Try import; if missing install from /kaggle/input wheels offline."""
    try:
        __import__(import_name)
        print(f"✅ Already available: {import_name}")
        return True
    except Exception:
        wheels = glob.glob(f"/kaggle/input/**/*{wheel_hint}*.whl", recursive=True)
        if not wheels:
            print(f"❌ Missing wheel for: {import_name} (hint={wheel_hint})")
            return False
        target = sorted(wheels, key=len)[0]
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", target, "--no-deps", "--ignore-installed", "--quiet"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
            reload(site)
            __import__(import_name)
            print(f"✅ Installed offline: {import_name}")
            return True
        except Exception as e:
            print(f"⚠️ Offline install failed for {import_name}: {e}")
            return False

# نضمن فقط المكتبات التي نحتاجها (بدون لمس torch)
ensure_pkg("timm", "timm")
ensure_pkg("segmentation_models_pytorch", "segmentation_models_pytorch")
ensure_pkg("ultralytics", "ultralytics")

# محرك parquet (عادة موجود في Kaggle)
try:
    import pyarrow  # noqa
    print("✅ pyarrow available (parquet OK)")
except Exception:
    # إذا مو موجود غالبًا Kaggle يوفره؛ لا نحاول تنزيله من النت
    print("⚠️ pyarrow not found. If parquet read fails later, Kaggle environment may be missing parquet engine.")

import torch
import segmentation_models_pytorch as smp

print("------ Environment Check ------")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("smp:", getattr(smp, "__version__", "unknown"))
print("✅ Cell 2 ready.")


🔧 Cell 2: Offline install/import fix (no internet).
✅ Already available: timm
✅ Installed offline: segmentation_models_pytorch
✅ Already available: ultralytics
✅ pyarrow available (parquet OK)
------ Environment Check ------
torch: 2.6.0+cu124
cuda available: True
gpu: Tesla T4
smp: 0.5.0
✅ Cell 2 ready.


In [3]:
# --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
DB_DIR = "physionet_db"
if not os.path.exists(DB_DIR):
    os.makedirs(DB_DIR)
    print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
    try:
        # تحميل عينة كافية لضمان التنوع
        records = wfdb.get_record_list('ptb-xl')
        selected_records = records[:200] 
        wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
        print(f"✅ تم تحميل {len(selected_records)} سجل.")
    except Exception as e:
        print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

class UltimateRenderer:
    def __init__(self, db_dir):
        self.db_dir = db_dir
        self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
    def get_real_signal(self):
        """سحب إشارة عشوائية من PTB-XL"""
        if not self.records:
            t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
        try:
            rec_name = random.choice(self.records)
            record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
            lead_idx = random.randint(0, record.shape[1] - 1)
            signal = record[:, lead_idx]
            signal = np.nan_to_num(signal)
            
            # قص عشوائي (Zoom in/out simulation)
            crop_len = random.randint(1000, 3000)
            if len(signal) > crop_len:
                start = random.randint(0, len(signal) - crop_len)
                return signal[start : start+crop_len]
            return signal
        except:
            return np.random.randn(2000)

    def render_to_memory(self, signal):
        """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
        # 3. الشبكة المتغيرة (Distractor)
        grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
        grid_alpha = random.uniform(0.3, 0.8)
        bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
        fig_h, fig_w = 3, 8
        dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
        # --- أ. توليد القناع (Mask Generation) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.plot(signal, color='white', linewidth=3.0) 
        ax.set_ylim(np.min(signal), np.max(signal))
        ax.axis('off')
        fig.patch.set_facecolor('black')
        
        fig.canvas.draw()
        mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
        plt.close(fig)

        # --- ب. الرسم الأولي (Rendering) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.set_facecolor(bg_color)
        
        # رسم الشبكة
        ax.minorticks_on()
        ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
        ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
        # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
        line_color = random.choice(['black', 'blue', '#000033']) 
        ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
        ax.axis('off')
        ax.set_ylim(np.min(signal), np.max(signal))
        fig.patch.set_facecolor(bg_color)
        
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
        plt.close(fig)
        
        return img, mask

print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")

⬇️ جاري تحميل سجلات PTB-XL الأساسية...
⚠️ تحذير: فشل التحميل (name 'wfdb' is not defined)، سيتم استخدام التوليد الاحتياطي.
✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).


In [4]:
# --- Cell 22: V41 Ultimate (Lead Map + Dynamic Len + Smart Grid Hypothesis + Safe) ---
import os, gc, csv, glob, math
import numpy as np
import pandas as pd
import cv2
import torch
from collections import OrderedDict
from tqdm import tqdm
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt
import segmentation_models_pytorch as smp
from ultralytics import YOLO

# =========================
# 0) CONFIG
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = "/kaggle/input/physionet-ecg-image-digitization"
TEST_CSV_PATH = f"{DATA_DIR}/test.csv"
SAMPLE_PARQUET_PATH = f"{DATA_DIR}/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

YOLO_PATH = "/kaggle/input/ecg-final-models/best.pt"
UNET_PATH = "/kaggle/input/ecg-final-models/best_model_phase10.pth"

LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
LEAD_TO_IDX = {n:i for i,n in enumerate(LEAD_NAMES)}

TARGET_H = 256
MAX_W = 2048
YOLO_CONF = 0.12
MAX_CACHE = 12

# Viterbi DP (أدق لكن أثقل). سيعمل فقط عندما العرض ليس كبيراً جداً.
USE_DP_VITERBI = True
DP_MAX_W = 1400     # فوق هذا، نرجع لـ argmax لتفادي البطء
DP_SMOOTH = 0.45

print(f"⚡ Device: {DEVICE}")

# =========================
# 1) TEMPLATE -> IDS + PATIENT LENGTHS
# =========================
if not os.path.exists(SAMPLE_PARQUET_PATH):
    raise FileNotFoundError("❌ sample_submission.parquet not found")

print("📦 Reading template ids...")
template = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
template_ids = template["id"].astype(str).to_numpy()
del template
gc.collect()
print(f"✅ Template rows: {len(template_ids):,}")

print("🧠 Scanning template for patient lengths...")
patient_lengths = {}
for rid in template_ids:
    # rid format: pid_idx_lead (pid may contain underscores)
    parts = rid.rsplit("_", 2)
    if len(parts) != 3:
        continue
    pid = str(parts[0]).strip()
    if pid.endswith(".0"): pid = pid[:-2]
    try:
        idx = int(parts[1])
    except:
        continue
    cur = patient_lengths.get(pid, 0)
    if idx + 1 > cur:
        patient_lengths[pid] = idx + 1
print(f"✅ Lengths mapped for {len(patient_lengths):,} patients.")

# =========================
# 2) INDEX IMAGES + FS MAP
# =========================
print("🗂️ Indexing images...")
image_paths = glob.glob(f"{DATA_DIR}/**/*.png", recursive=True) + glob.glob(f"{DATA_DIR}/**/*.jpg", recursive=True)
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
print(f"✅ Indexed images: {len(path_map):,}")

fs_map = {}
if os.path.exists(TEST_CSV_PATH):
    try:
        tdf = pd.read_csv(TEST_CSV_PATH, dtype=str)
        cols_low = {c.lower(): c for c in tdf.columns}
        id_c = next((cols_low[c] for c in cols_low if "id" in c), None)
        fs_c = next((cols_low[c] for c in cols_low if "fs" in c), None)
        if id_c and fs_c:
            fs_map = dict(zip(tdf[id_c].astype(str), tdf[fs_c].astype(str)))
            print(f"✅ fs_map loaded: {len(fs_map):,} items")
    except Exception as e:
        print(f"⚠️ fs_map read failed: {e}")

# =========================
# 3) LOAD MODELS + LEAD MAPPING
# =========================
print("⚙️ Loading models (offline)...")

# YOLO + class-name mapping
yolo_model = None
CLASS_TO_LEAD_IDX = {}

if os.path.exists(YOLO_PATH):
    try:
        yolo_model = YOLO(YOLO_PATH)
        print(f"✅ YOLO loaded: {YOLO_PATH}")
        # Build mapping from names if present
        if hasattr(yolo_model, "names") and isinstance(yolo_model.names, dict):
            for cid, cname in yolo_model.names.items():
                n = str(cname).strip().replace("Lead", "").replace("lead", "").replace(" ", "")
                n = n.replace("AVR","aVR").replace("AVL","aVL").replace("AVF","aVF")
                if n in LEAD_TO_IDX:
                    CLASS_TO_LEAD_IDX[int(cid)] = LEAD_TO_IDX[n]
        # Fallback identity
        for i in range(12):
            CLASS_TO_LEAD_IDX.setdefault(i, i)
        print(f"✅ CLASS_TO_LEAD_IDX: {CLASS_TO_LEAD_IDX}")
    except Exception as e:
        print(f"⚠️ YOLO load failed: {e}")

# UNet
unet_model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    decoder_attention_type="scse"
)
if os.path.exists(UNET_PATH):
    try:
        unet_model.load_state_dict(torch.load(UNET_PATH, map_location=DEVICE))
        print(f"✅ UNet loaded: {UNET_PATH}")
    except Exception as e:
        print(f"⚠️ UNet load failed: {e}")

unet_model.to(DEVICE)
unet_model.eval()

# =========================
# 4) HELPERS
# =========================
def clean_pid(pid):
    s = str(pid).strip()
    return s[:-2] if s.endswith(".0") else s

def get_image_path(pid):
    pid_s = clean_pid(pid)
    p = path_map.get(pid_s)
    if p is None:
        try:
            p = path_map.get(str(int(float(pid_s))))
        except:
            return None
    return p

def safe_read_rgb(path):
    try:
        img = cv2.imread(path)
        if img is None:
            return None
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    except:
        return None

def remove_red_grid(img):
    # lightweight grid removal; safe if fails
    try:
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        mask = (cv2.inRange(hsv, (0, 50, 50), (10, 255, 255)) |
                cv2.inRange(hsv, (170, 50, 50), (180, 255, 255)))
        out = img.copy()
        out[mask > 0] = 255
        return out
    except:
        return img

def robust_grid_spacing_px(img, default_val=25.0):
    """
    محاولة تقدير المسافة بين خطوط الشبكة بالبكسل.
    قد ترجع 1mm spacing أو 5mm spacing حسب الصورة.
    """
    try:
        if img is None or img.size == 0:
            return float(default_val)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        # edge energies in both axes
        ex = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        ey = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        sx = np.sum(np.abs(ex), axis=0)
        sy = np.sum(np.abs(ey), axis=1)

        # find peaks
        px, _ = find_peaks(sx, distance=8, prominence=np.percentile(sx, 75))
        py, _ = find_peaks(sy, distance=8, prominence=np.percentile(sy, 75))

        cand = []
        if len(px) > 6:
            dx = np.diff(px)
            cand.extend(dx[(dx > 4) & (dx < 180)])
        if len(py) > 6:
            dy = np.diff(py)
            cand.extend(dy[(dy > 4) & (dy < 180)])

        if len(cand) >= 8:
            return float(np.median(cand))
    except:
        pass
    return float(default_val)

def butter_highpass(data, fs, cutoff=0.5, order=3):
    try:
        if fs <= 1:
            return data
        nyq = 0.5 * fs
        w = cutoff / nyq
        if w <= 0 or w >= 1:
            return data
        b, a = butter(order, w, btype='high')
        return filtfilt(b, a, data)
    except:
        return data

def butter_lowpass(data, fs, cutoff=40.0, order=3):
    try:
        if fs <= 1:
            return data
        nyq = 0.5 * fs
        w = cutoff / nyq
        if w <= 0 or w >= 1:
            return data
        b, a = butter(order, w, btype='low')
        return filtfilt(b, a, data)
    except:
        return data

def robust_p2p(sig):
    if sig is None or len(sig) == 0:
        return 0.0
    p95 = np.percentile(sig, 95)
    p05 = np.percentile(sig, 5)
    return float(p95 - p05)

def choose_ppmv_and_convert(raw_px, grid_px, scale_resized):
    """
    تحويل raw_px (بكسل في صورة 256) إلى mV باختيار فرضية الشبكة الأفضل:
    - فرضية A: grid_px هي 1mm  -> pixels_per_mV = grid_px * 10
    - فرضية B: grid_px هي 5mm  -> pixels_per_mV = grid_px * 2
    ثم نضرب بـ scale_resized لأن raw_px في صورة مُعاد تحجيمها.
    """
    # sanitize
    g = float(grid_px) if grid_px and np.isfinite(grid_px) else 25.0
    s = float(scale_resized) if scale_resized and np.isfinite(scale_resized) else 1.0
    g = max(3.0, min(g, 120.0))
    s = max(0.05, min(s, 20.0))

    ppmv_small = (g * 10.0) * s  # if g ~= 1mm
    ppmv_big   = (g *  2.0) * s  # if g ~= 5mm

    ppmv_small = max(15.0, min(ppmv_small, 5000.0))
    ppmv_big   = max(15.0, min(ppmv_big,   5000.0))

    centered = raw_px - np.median(raw_px)

    sig_small = centered / ppmv_small
    sig_big   = centered / ppmv_big

    # Choose by plausibility (بدون تطبيع هجومي)
    p2p_s = robust_p2p(sig_small)
    p2p_b = robust_p2p(sig_big)

    # target typical p2p range (conservative)
    ok_s = (0.15 <= p2p_s <= 10.0)
    ok_b = (0.15 <= p2p_b <= 10.0)

    if ok_s and not ok_b:
        return sig_small
    if ok_b and not ok_s:
        return sig_big

    # If both ok or both not ok -> choose closer to typical ~2.0mV (but no forced scaling)
    target = 2.0
    ds = abs(p2p_s - target)
    db = abs(p2p_b - target)
    return sig_small if ds <= db else sig_big

def viterbi_dp(prob_map, smooth=0.45):
    """
    DP Viterbi (local transitions). Accurate but heavier.
    """
    H, W = prob_map.shape
    cost = -np.log(prob_map + 1e-6).astype(np.float32)

    dp = np.zeros_like(cost)
    parent = np.zeros((H, W), dtype=np.int16)
    dp[:, 0] = cost[:, 0]

    for t in range(1, W):
        prev = dp[:, t-1]
        c_m1 = np.roll(prev, 1);  c_m1[0]  = 1e9
        c_0  = prev
        c_p1 = np.roll(prev, -1); c_p1[-1] = 1e9

        stacked = np.stack([c_m1 + smooth, c_0, c_p1 + smooth], axis=0)
        which = np.argmin(stacked, axis=0)  # 0,1,2
        dp[:, t] = cost[:, t] + stacked[which, np.arange(H)]
        parent[:, t] = np.clip(np.arange(H) + (which - 1), 0, H-1)

    path = np.zeros(W, dtype=np.int32)
    path[-1] = int(np.argmin(dp[:, -1]))
    for t in range(W-2, -1, -1):
        path[t] = parent[path[t+1], t+1]

    return (H - path).astype(np.float32)

def extract_path(prob_map):
    H, W = prob_map.shape
    if USE_DP_VITERBI and W <= DP_MAX_W:
        return viterbi_dp(prob_map, smooth=DP_SMOOTH)
    # fast: argmax + smoothing
    path = np.argmax(prob_map, axis=0)
    if W >= 15:
        win = 11 if W >= 11 else (W//2)*2+1
        path = savgol_filter(path, win, 2)
    return (H - path).astype(np.float32)

def get_crops_mapped(img, model):
    h, w = img.shape[:2]
    crops = [None] * 12

    # grid fallback
    rh, cw = h // 4, w // 3
    grid = [img[r*rh:(r+1)*rh, c*cw:(c+1)*cw] for r in range(4) for c in range(3)]

    if model is not None:
        try:
            scale = 1280 / max(h, w)
            if scale < 1.0:
                img_inf = cv2.resize(img, (int(w*scale), int(h*scale)))
            else:
                img_inf = img
                scale = 1.0

            res = model.predict(img_inf, conf=YOLO_CONF, verbose=False, device=DEVICE)
            if res and res[0].boxes:
                for box in res[0].boxes.data.detach().cpu().numpy():
                    cls = int(box[5])
                    tgt = CLASS_TO_LEAD_IDX.get(cls, cls)
                    if 0 <= tgt < 12:
                        x1, y1, x2, y2 = box[:4] / scale
                        x1, y1 = max(0, int(x1)), max(0, int(y1))
                        x2, y2 = min(w, int(x2)), min(h, int(y2))
                        if x2 > x1+10 and y2 > y1+10:
                            crops[tgt] = img[y1:y2, x1:x2]
        except:
            pass

    for i in range(12):
        if crops[i] is None:
            crops[i] = grid[i]
    return crops

# =========================
# 5) MAIN COMPUTE (per patient)
# =========================
patient_cache = OrderedDict()

def compute_patient(pid, target_len):
    out = np.zeros((12, target_len), dtype=np.float32)
    path = get_image_path(pid)
    if not path:
        return out

    img = safe_read_rgb(path)
    if img is None:
        return out

    fs = float(fs_map.get(clean_pid(pid), 500.0))
    fs = 500.0 if (not np.isfinite(fs) or fs <= 0) else fs

    try:
        crops = get_crops_mapped(img, yolo_model)

        # global grid fallback
        global_grid = robust_grid_spacing_px(img, default_val=25.0)

        # prepare batch
        inputs = []
        scales = []
        clean_crops = []

        for c in crops:
            c2 = remove_red_grid(c)
            clean_crops.append(c2)

            h0, w0 = c2.shape[:2]
            if h0 < 5 or w0 < 5:
                # invalid crop
                inputs.append(torch.zeros((3, TARGET_H, 32), dtype=torch.float32))
                scales.append(1.0)
                continue

            s = TARGET_H / float(h0)
            nw = int(max(32, w0 * s))
            if nw > MAX_W:
                nw = MAX_W
            rz = cv2.resize(c2, (nw, TARGET_H))
            t = torch.from_numpy(rz).permute(2,0,1).float() / 255.0
            inputs.append(t)
            scales.append(s)

        max_w = max(t.shape[2] for t in inputs)
        max_w = int(np.ceil(max_w / 32) * 32)

        batch = torch.zeros((12, 3, TARGET_H, max_w), dtype=torch.float32, device=DEVICE)
        for i, t in enumerate(inputs):
            ww = min(t.shape[2], max_w)
            batch[i, :, :, :ww] = t[:, :, :ww].to(DEVICE)

        with torch.no_grad():
            # autocast for speed
            with torch.amp.autocast('cuda', enabled=(DEVICE=='cuda')):
                pred = torch.sigmoid(unet_model(batch)).detach().float().cpu().numpy()

        for i in range(12):
            w_i = inputs[i].shape[2]
            prob = pred[i, 0, :, :w_i]

            raw_px = extract_path(prob)

            # local grid or fallback global
            local_grid = robust_grid_spacing_px(clean_crops[i], default_val=global_grid)
            if not np.isfinite(local_grid) or local_grid < 3:
                local_grid = global_grid

            # convert to mV using best grid hypothesis
            sig_mv = choose_ppmv_and_convert(raw_px, local_grid, scales[i])

            # baseline center + filters (light, not destructive)
            sig_mv = sig_mv - np.median(sig_mv)
            sig_mv = np.nan_to_num(sig_mv, nan=0.0, posinf=0.0, neginf=0.0)

            # mild band shaping
            sig_mv = butter_highpass(sig_mv, fs, cutoff=0.5, order=3)
            sig_mv = butter_lowpass(sig_mv, fs, cutoff=40.0, order=3)

            # clip extreme outliers only
            sig_mv = np.clip(sig_mv, -8.0, 8.0)

            # resample to required length
            if len(sig_mv) > 8:
                out[i] = resample(sig_mv.astype(np.float32), target_len).astype(np.float32)

        # Einthoven soft consistency: II ≈ I + III
        # blend gently (لا نكسر الإشارة)
        out[1] = 0.6*out[1] + 0.4*(out[0] + out[2])

        return out

    except RuntimeError as e:
        # OOM or similar
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            return out
        raise
    except:
        torch.cuda.empty_cache()
        return out

# =========================
# 6) STREAM WRITE SUBMISSION
# =========================
print("🚀 Writing submission.csv (V41 Ultimate)...")

with open(SUBMISSION_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "value"])

    for rid in tqdm(template_ids, desc="Processing Rows"):
        try:
            parts = rid.rsplit("_", 2)
            if len(parts) != 3:
                writer.writerow([rid, "0.0000"])
                continue

            pid = clean_pid(parts[0])
            idx = int(parts[1])
            lead = parts[2]

            t_len = patient_lengths.get(pid, 5000)

            if pid in patient_cache:
                mat = patient_cache[pid]
                patient_cache.move_to_end(pid)
            else:
                mat = compute_patient(pid, t_len)
                patient_cache[pid] = mat
                if len(patient_cache) > MAX_CACHE:
                    patient_cache.popitem(last=False)

            lidx = LEAD_TO_IDX.get(lead, 0)

            val = 0.0
            if 0 <= idx < t_len:
                v = float(mat[lidx][idx])
                if np.isfinite(v):
                    val = v

            writer.writerow([rid, f"{val:.4f}"])

        except:
            writer.writerow([rid, "0.0000"])

del patient_cache
gc.collect()
torch.cuda.empty_cache()

print("✅ Done. submission.csv ready.")
print("🎉 Ready to Submit.")


⚡ Device: cuda
📦 Reading template ids...
✅ Template rows: 75,000
🧠 Scanning template for patient lengths...
✅ Lengths mapped for 2 patients.
🗂️ Indexing images...
✅ Indexed images: 8,795
✅ fs_map loaded: 2 items
⚙️ Loading models (offline)...
✅ YOLO loaded: /kaggle/input/ecg-final-models/best.pt
✅ CLASS_TO_LEAD_IDX: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11}
✅ UNet loaded: /kaggle/input/ecg-final-models/best_model_phase10.pth
🚀 Writing submission.csv (V41 Ultimate)...


Processing Rows: 100%|██████████| 75000/75000 [00:03<00:00, 19665.90it/s]


✅ Done. submission.csv ready.
🎉 Ready to Submit.


In [5]:
# =========================
# --- الخلية 23: Strict Submission Validator (Must Pass) ---
# =========================
import os
import numpy as np
import pandas as pd

SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

print("🔍 Validating submission.csv strictly...")

if not os.path.exists(SUBMISSION_FILE):
    raise FileNotFoundError("❌ submission.csv not found. Run Cell 22 first.")

# اقرأ القالب (ids فقط)
tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
tmpl_ids = tmpl["id"].astype(str).to_numpy()
n_tmpl = len(tmpl_ids)

# اقرأ ملفك
sub = pd.read_csv(SUBMISSION_FILE)

# 1) الأعمدة
assert list(sub.columns) == ["id", "value"], f"❌ Columns mismatch: {sub.columns}"

# 2) عدد الصفوف
assert len(sub) == n_tmpl, f"❌ Row count mismatch: sub={len(sub)} vs template={n_tmpl}"

# 3) عدم وجود NaN
assert sub["id"].isna().sum() == 0, "❌ Found NaN in id"
assert sub["value"].isna().sum() == 0, "❌ Found NaN in value"

# 4) تحويل value إلى float والتأكد finite
vals = pd.to_numeric(sub["value"], errors="coerce").to_numpy()
assert np.isfinite(vals).all(), "❌ Found non-finite values (NaN/inf) in value"

# 5) مطابقة أول وآخر ID (كفاية لاكتشاف تغيير ترتيب/تنظيف خاطئ)
sub_ids = sub["id"].astype(str).to_numpy()
assert sub_ids[0] == tmpl_ids[0], f"❌ First ID mismatch: {sub_ids[0]} != {tmpl_ids[0]}"
assert sub_ids[-1] == tmpl_ids[-1], f"❌ Last ID mismatch: {sub_ids[-1]} != {tmpl_ids[-1]}"

# 6) فحص عينة عشوائية للمطابقة (بدون مقارنة كل شيء لتوفير وقت)
idxs = np.linspace(0, n_tmpl-1, num=min(20, n_tmpl), dtype=int)
for i in idxs:
    if sub_ids[i] != tmpl_ids[i]:
        raise AssertionError(f"❌ ID mismatch at row {i}: {sub_ids[i]} != {tmpl_ids[i]}")

size_mb = os.path.getsize(SUBMISSION_FILE) / (1024*1024)
print("✅ All checks passed.")
print(f"📦 submission.csv size: {size_mb:.2f} MB")
print("🎉 Ready to Submit.")


🔍 Validating submission.csv strictly...
✅ All checks passed.
📦 submission.csv size: 1.96 MB
🎉 Ready to Submit.
